# 03 — Feature Engineering

**IBM Bob assisted** — All feature engineering code generated via IBM Bob Phase 3.

Produces: rolling averages · lag features · flare flags · CME arrival score · cyclical encodings → `data/processed/features_v1.parquet` + `FEATURE_PROVENANCE.json`

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from dashboard.components.features import build_feature_matrix, save_features

RAW_DIR  = Path('../data/raw')
PROC_DIR = Path('../data/processed')

plt.rcParams['figure.dpi'] = 120
print('Imports OK')

In [ ]:
# Load daily master (produced by notebook 02)
master_path = PROC_DIR / 'daily_master.parquet'
if master_path.exists():
    master = pd.read_parquet(master_path)
    print(f'Loaded daily_master: {master.shape}')
    display(master.head())
else:
    print('daily_master.parquet not found — building synthetic data for demonstration')
    rng = np.random.default_rng(42)
    dates = pd.date_range('2023-01-01', '2024-08-30', freq='D', tz='UTC')
    master = pd.DataFrame({
        'date': dates,
        'kp_mean': rng.uniform(0, 9, len(dates)),
        'f10_7':   rng.uniform(70, 220, len(dates)),
        'launch_go': np.where(rng.uniform(0,1,len(dates)) < 0.7, 1, 0),
        'launch_window_utc': [f'{rng.integers(0,24):02d}:00' for _ in dates],
    })
    # Only ~52 launch events, rest NaN
    mask = rng.random(len(master)) > 0.86
    master.loc[~mask, 'launch_go'] = np.nan
    master.loc[~mask, 'launch_window_utc'] = np.nan
    print(f'Synthetic master: {master.shape}')

In [ ]:
# Load optional event dataframes
def _load_donki(event: str):
    files = sorted(RAW_DIR.glob(f'donki_{event.lower()}_*.json'), reverse=True)
    if not files:
        return None
    with open(files[0]) as f:
        data = json.load(f)
    return pd.DataFrame(data) if data else None

flr_df = _load_donki('FLR')
cme_df = _load_donki('CME')
gst_df = _load_donki('GST')
print('FLR:', None if flr_df is None else len(flr_df),
      '  CME:', None if cme_df is None else len(cme_df),
      '  GST:', None if gst_df is None else len(gst_df))

In [ ]:
# Build feature matrix
features = build_feature_matrix(master, flr_df=flr_df, cme_df=cme_df, gst_df=gst_df)
print(f'Feature matrix shape: {features.shape}')
print('Columns:', features.columns.tolist())
display(features.head(10))

In [ ]:
# Null check on features
null_pct = (features.isnull().sum() / len(features) * 100).round(2)
print('Null % per column:')
print(null_pct[null_pct > 0].to_string() or '  (no nulls)')

In [ ]:
# Feature distribution plots
num_features = [c for c in features.columns if c not in ('date','launch_go')]
n_cols = 4
n_rows = (len(num_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3))
axes = axes.flatten()
for i, col in enumerate(num_features):
    axes[i].hist(features[col].dropna(), bins=30, color='#0f62fe', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')
for j in range(len(num_features), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Feature Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROC_DIR / 'feature_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation with launch_go
if 'launch_go' in features.columns:
    labeled = features.dropna(subset=['launch_go'])
    print(f'Labeled rows: {len(labeled)}  (GO={int(labeled["launch_go"].sum())}, SCRUB={int((labeled["launch_go"]==0).sum())})')
    corr = labeled[num_features + ['launch_go']].corr()['launch_go'].drop('launch_go').sort_values()
    fig, ax = plt.subplots(figsize=(8, 6))
    corr.plot(kind='barh', ax=ax, color=['crimson' if v < 0 else '#0f62fe' for v in corr])
    ax.set_title('Feature Correlation with launch_go', fontsize=13)
    ax.set_xlabel('Pearson r')
    ax.axvline(0, color='black', lw=0.8)
    plt.tight_layout()
    plt.savefig(PROC_DIR / 'feature_target_correlation.png', bbox_inches='tight')
    plt.show()

In [ ]:
# Save features + FEATURE_PROVENANCE.json
provenance = save_features(features, version=1)
print(json.dumps(provenance, indent=2))